# Image Colorization with Pre-trained CNN Autoencoder

In [ ]:
import cv2
import numpy as np
from skimage.color import rgb2lab, lab2rgb
import warnings
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
import tensorflow as tf
warnings.filterwarnings('ignore')

### Model: 
- **Architecture**: Convolutional autoencoder (Encoder–Decoder) with MaxPooling2D for compression and UpSampling2D for reconstruction.  
- **Output**: two channels (a, b) with final `tanh` activation (values in range −1..1).  
- **Load & inspect**: `model = tf.keras.models.load_model(MODEL_PATH)` then `model.summary()` to view layers and parameter counts.  

In [ ]:
MODEL_PATH = "../Colorized-Img/colorize_autoencoder_first1000.keras"

IMAGES_PATHS = [
    "../images/forest.397.jpg",
    "../images/forest.399.jpg",
    "../images/YIN8QU2CC97X.jpg",
    "../images/ZYLOMHW6DWLE.jpg"
]
IMAGE_NAMES = ['Forest 1', 'Forest 2', 'Mountain', 'Barn']

TARGET = 256

In [ ]:
model = load_model(MODEL_PATH)
model.summary()

## Grayscale Conversion to Lab Color Space

**Preprocessing (required for both methods)**:  
- **Resize** each image to `256×256`.  
- **Method 1 (stack grayscale → rgb → Lab → L)**:  
  - Duplicate the single-channel grayscale image across the channel dimension to create a 3-channel image.  
  - Convert with `skimage.color.rgb2lab(three_channel_image)` and extract the L channel.  
- **Method 2 (cv2 IMREAD_COLOR trick)**:  
  - Load the grayscale file with `cv2.imread(path, cv2.IMREAD_COLOR)` (OpenCV produces 3 channels).  
  - Convert the resulting BGR→RGB (if using OpenCV) then `rgb2lab` and extract L.  
- **Normalization**: Model expects L in range [0,100] before any scaling applied by training — confirm and normalize exactly as used during model training (e.g., scale to [0,1] or other as required).   
- **Final Assessment**: Both implementations successfully satisfy the requirement for a $(256, 256, 1)$ input tensor. Nevertheless, Method 1 represents the preferred architectural choice, as it achieves the same output integrity while eliminating the performance bottleneck inherent in the encoding-decoding sequence utilized in Method 2.

In [ ]:
processed_data = []

for path in IMAGES_PATHS:
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (TARGET, TARGET))
    
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    gray_3ch = np.stack((gray,) * 3, axis=-1)
    
    lab = rgb2lab(gray_3ch.astype(float) / 255.0)
    
    L = lab[:, :, 0:1]
    
    processed_data.append(L)

print("Method 1 - Feature Shapes:")
for idx, data in enumerate(processed_data):
    print(f"Index {idx} tensor shape: {data.shape}")

In [ ]:
processed_data_v2 = []

for path in IMAGES_PATHS:
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (256,256))
    
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    gray_rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)    
    
    lab = rgb2lab(gray_rgb.astype(float) / 255.0)
    
    L = lab[:,:,0:1]
    
    processed_data_v2.append(L)

print("Method 2 - Output Shapes:")
for i, data in enumerate(processed_data_v2):
    print(f"Index {i} L-channel shape: {data.shape}")
    

## Image Reconstruction and Colorization Results

**Inference & Reconstruction**:  
- Feed the prepared `L` (shape `256×256×1`) into `model.predict()` to obtain predicted `ab` with range −1..1.  
- Rescale predicted `ab` to the Lab `a/b` range used by training if necessary.  
- Concatenate original L and predicted `ab` to form Lab image, then convert back to RGB using `skimage.color.lab2rgb`. 


## Analysis of Methods 1 vs. 2
Looking at your first image, we can see that Method 1 and Method 2 produce nearly identical outputs.

- **Why?** As suspected, both methods provide the model with the exact same input information. Both `np.stack((gray,) * 3)` and `cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)` create a 3-channel image where `𝑅=G=B`. 
- **Colorization Quality**: The model is producing a “plausible” rather than “perfect” colorization. It seems to have correctly identified that the foreground (the grass and tree base) should be brownish/green, and it has introduced some warm (reddish/orange) hues in the lower left, which might be an attempt to colorize based on patterns it learned during training.

In [ ]:
def apply_colorization(l_channel, model):
    l_input = np.expand_dims(l_channel, axis=0)
    ab_preds = model.predict(l_input, verbose=0)[0] * 128.0
    return lab2rgb(np.concatenate((l_channel, ab_preds), axis=2))

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 20))
fig.suptitle("Comparative Analysis: Colorization Pipelines", fontsize=16)

for idx in range(len(IMAGES_PATHS)):
    rgb_m1 = apply_colorization(processed_data[idx], model)
    rgb_m2 = apply_colorization(processed_data_v2[idx], model)
    
    img = plt.imread(IMAGES_PATHS[idx])
    
    axes[idx, 0].imshow(img)
    axes[idx, 0].set_title(f"Original: {IMAGE_NAMES[idx]}")
    axes[idx, 0].axis('off')

    axes[idx, 1].imshow(rgb_m1)
    axes[idx, 1].set_title(f"Method 1: {IMAGE_NAMES[idx]}")
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(rgb_m2)
    axes[idx, 2].set_title(f"Method 2: {IMAGE_NAMES[idx]}")
    axes[idx, 2].axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## Analysis of the Rotation Test
Second image demonstrates the spatial sensitivity of CNN:

- **Observation**: The “Rotation Test” result does not look identical to the “Without Rotation” result.
- **Why?** CNNs are not inherently rotation-invariant. The model learned to colorize specific features (like trees and ground) based on their vertical orientation in the training images. When we rotated the input by 45 degrees, the model saw a “sideways” landscape. The convolutional kernels, which are tuned for horizontal/vertical edges, interpreted the rotated features differently, leading to a different color output. When we rotated it back, those “wrongly colored” pixels remained, creating the visual difference we see between the two plots.


In [ ]:
img_path = IMAGES_PATHS[0]

img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (TARGET, TARGET))

gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

center = (128, 128)

M = cv2.getRotationMatrix2D(center, 45, 1.0)

rotated = cv2.warpAffine(gray, M, (256,256))

rotated_rgb_3ch = np.stack((rotated,) * 3, axis=-1)

lab_rotated = rgb2lab(rotated_rgb_3ch.astype(float) / 255.0)

L_rotated = lab_rotated[:,:,0:1]

rotated_result = apply_colorization(L_rotated, model)

M_back = cv2.getRotationMatrix2D(center, -45, 1.0)

aligned = cv2.warpAffine(
    (rotated_result*255).astype(np.uint8),
    M_back,
    (256,256)
)

gray_rgb = np.stack((gray,) * 3, axis=-1)
lab_original = rgb2lab(gray_rgb.astype(float) / 255.0)
L_original = lab_original[:,:,0:1]

normal_result = apply_colorization(L_original, model)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(normal_result)
plt.title("Without Rotation")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(aligned)
plt.title("Rotation Test")
plt.axis("off")

plt.show()

## Analysis of Gaussian Noise
- **Observation**: The output is stable and I guess this is because of little sigma chosen for noise. While the image is slightly “grainier,” the overall color distribution remains largely intact compared to the original output.
- **Why?** Convolutional Neural Networks are generally good at filtering out low-level noise. Since autoencoder was trained to map L (luminance) to ab (color), the deeper layers are effectively “averaging” the input features. Small, random pixel-level noise does not significantly shift the global feature extraction that determines color.
- **Affected Areas**: We notice the most change in uniform, low-texture areas (like the sky or foggy sections), where the noise disrupts the smooth luminance gradients, causing minor, mottled color shifts. If we increase sigma to for exmaple 5.0, we will see difference and it is more obviuse on areas with high-frequency like trees texture, and etc.

In [ ]:
noise = np.random.normal(0, 0.1, L_original.shape)

L_noise = L_original + noise

L_noise = np.clip(L_noise, 0, 100)

noise_result = apply_colorization(L_noise, model)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(normal_result)
plt.title("Original Output")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(noise_result)
plt.title("Gaussian Noise")
plt.axis("off")

plt.show()

In [ ]:
noise = np.random.normal(0, 3, L_original.shape)

L_noise = L_original + noise

L_noise = np.clip(L_noise, 0, 100)

noise_result = apply_colorization(L_noise, model)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(normal_result)
plt.title("Original Output")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(noise_result)
plt.title("Gaussian Noise")
plt.axis("off")

plt.show()

## Analysis of Brightness (Stability)
- **Observation**: The colors do not change proportionally to the brightness.
- **Brightness 0.7 (Darker)**: The colors become more saturated, and the model tends to favor deeper, more intense shades. Because the model is looking for patterns in a darker input, it may be misinterpreting low-light noise or textures as objects that should be colorized differently.
- **Brightness 1.3 (Brighter)**: The colors become washed out. In many areas, the model loses the ability to distinguish “features” because the luminance range is pushed toward pure white (clipping), forcing the model to output a more desaturated, “greyish” or “pastel” color palette.
- **Conclusion**: This confirms that the model is highly dependent on the input luminance context. It did not learn “this object is always green”; it learned “this luminance pattern usually corresponds to this color.” When you change the luminance, you break the relationship the model learned, causing the colorization to shift in non-linear, often unpredictable ways.

In [ ]:
L_dark = np.clip(L_original * 0.7, 0, 100)

L_bright = np.clip(L_original * 1.3, 0, 100)

dark_result = apply_colorization(L_dark, model)
bright_result = apply_colorization(L_bright, model)

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(normal_result)
plt.title("Original")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(dark_result)
plt.title("Brightness x0.7")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(bright_result)
plt.title("Brightness x1.3")
plt.axis("off")

plt.show()